In [ ]:
"""Graph node counting task."""

from fundl.datasets import make_graph_counting_dataset
from fundl.utils import pad_graph
import numpy as onp
import networkx as nx
import jax.numpy as np

import panel as pn
import param
import holoviews as hv
import hvplot
import hvplot.streamz

# hv.extension("bokeh")

In [ ]:
from streamz.dataframe import DataFrame
from streamz import Stream

In [ ]:
# pn.extension()

In [ ]:
Gs = make_graph_counting_dataset(n_graphs=1000)
Fs = []
As = []
for G in Gs:
    Fs.append(onp.vstack([d["features"] for n, d in G.nodes(data=True)]))
    As.append(onp.asarray(nx.adjacency_matrix(G).todense()))

In [ ]:
largest_graph_size = max([len(G) for G in Gs])

y = np.array([len(G) for G in Gs]).reshape(-1, 1)
y_train = y[0:800, :]
y_test = y[800:, :]

for i, (F, A) in enumerate(zip(Fs, As)):
    F, A = pad_graph(F, A, largest_graph_size)
    Fs[i] = F
    As[i] = A

Fs = onp.stack(Fs).astype(float)
As = onp.stack(As).astype(float)

Fs_train = Fs[0:800, :, :]
As_train = As[0:800, :, :]

Fs_test = Fs[800:, :, :]
As_test = As[800:, :, :]

print(Fs.shape)
print(As.shape)

In [ ]:
from fundl.weights import add_dense_params

params = dict()
params = add_dense_params(params, name="graph1", input_dim=2, output_dim=5)
params = add_dense_params(params, name="graph2", input_dim=5, output_dim=3)
params = add_dense_params(params, name="dense1", input_dim=3, output_dim=1)
# params = add_dense_params(params, name="dense2")

In [ ]:
from fundl.layers.graph import mpnn, gather
from fundl.layers import dense
from fundl.activations import relu
from fundl.losses import _mse_loss
from jax import grad

def mseloss(p, model, Fs, As, y):
    yhat = model(p, Fs, As)
    return np.mean(_mse_loss(y, yhat))


def model(params, Fs, As):
    Fs = mpnn(params["graph1"], As, Fs, nonlin=relu)
    Fs = mpnn(params["graph2"], As, Fs, nonlin=relu)
    out = gather(Fs)
    out = dense(params["dense1"], out, nonlin=relu)
    return out


dloss = grad(mseloss)

In [ ]:
import pandas as pd

class LossViewer(param.Parameterized):
    stream = Stream()

    example = pd.DataFrame(dict(iteration=[], loss=[]))
    sdf = DataFrame(stream, example=example)

    def plot_stream(self):
        return self.sdf.hvplot.line(x="iteration", y="loss").opts(logy=True)
    
    def emit(self, i, l):
        df = pd.DataFrame(dict(iteration=[i], loss=[l]))
        self.stream.emit(df)

In [ ]:
lv_train = LossViewer()
lv_test = LossViewer()
lv_train.plot_stream() * lv_test.plot_stream()

In [ ]:
from jax.experimental.optimizers import adam

init, update, get_params = adam(step_size=0.005)


state = init(params)
for i in range(1000):
    g = dloss(params, model, Fs_train, As_train, y_train)
    l_test = mseloss(params, model, Fs_test, As_test, y_test)
    l_train = mseloss(params, model, Fs_train, As_train, y_train)
    

    state = update(i, g, state)
    params = get_params(state)
    preds = model(params, Fs, As)
    print(l_test, l_train)
#     lv_train.emit(float(i), float(l_train))
#     lv_test.emit(float(i), float(l_test))
    # streamingdf.append(float(l))
